In [ ]:
import sys,os
__script_path=os.path.abspath(globals().get('__file__','.'))
__script_dir = os.path.dirname(__script_path)
root_dir = os.path.abspath(f'{__script_dir}/..')
print(root_dir)
for lib in [root_dir][::-1]:
    if lib in sys.path:
        sys.path.remove(lib)
    sys.path.insert(0,lib)


In [ ]:
from configs.config import *
from libs.common import *
from utils.format_utils import *
# from utils.extract_tables import full_pipeline
from utils.rag_evaluation import *
from utils.rag_qdrant_utils import *
load_dotenv(find_dotenv())

In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/Admin/Data/WDM-AI-TEMIS/serene-craft-464519-j1-963d43d7e20e.json"


In [ ]:
from PIL import Image
import requests
from IPython.display import display, HTML
import base64
from io import BytesIO


def show_top_images_from_urls(image_urls: list[str]):
    """
    Hiển thị hình ảnh từ danh sách URL ảnh.
    Dùng cho kết quả từ `search_image_urls(...)`

    Args:
        image_urls (list[str]): Danh sách link ảnh GCP hoặc internet
    """

    html = ""
    for url in image_urls:
        try:
            # Tải ảnh
            response = requests.get(url)
            img = Image.open(BytesIO(response.content)).convert("RGB")

            # Encode base64 để hiển thị inline
            buffered = BytesIO()
            img.save(buffered, format="JPEG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

            html += f"""
            <div style="display:inline-block;margin:10px;text-align:center">
                <img src="data:image/jpeg;base64,{img_base64}" style="width:200px;border-radius:8px;box-shadow:0px 2px 6px rgba(0,0,0,0.3)">
                <div style="margin-top:5px;max-width:200px;word-wrap:break-word;font-size:12px">
                    <i>{url.split('/')[-1]}</i>
                </div>
            </div>
            """

        except Exception as e:
            print(f"⚠️ Lỗi khi tải {url}: {e}")

    display(HTML(html))


In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import requests
from io import BytesIO
import torch

# Load VLM2Vec model
processor = AutoProcessor.from_pretrained("VLM2Vec/VLM2Vec-V2.0")
vlmmodel = AutoModel.from_pretrained("VLM2Vec/VLM2Vec-V2.0").to("cuda").eval()


In [ ]:
##Uncomment to initialise qdrant client in memory
# client.close()
client = qdrant_client.QdrantClient(
   path=f"{exps_dir}/qdrant_client_memory",
)

# ##Uncomment below to connect to Qdrant Cloud
# client = qdrant_client.QdrantClient(
#     os.environ.get("QDRANT_URL"),
#     api_key=os.environ.get("QDRANT_API_KEY"),
# )

## Uncomment below to connect to local Qdrant
#client = qdrant_client.QdrantClient("http://localhost:6333")

In [ ]:
from google.cloud import storage
from google.api_core.exceptions import Conflict, NotFound

import os

def upload_images_to_gcp(bucket_name: str, image_dir: str, location: str = "asia-southeast1") -> dict:
    storage_client = storage.Client()
    try:
        bucket = storage_client.get_bucket(bucket_name)
        print(f"[+] Bucket '{bucket_name}' đã tồn tại.")
        return bucket
    except NotFound:
        print(f"[-] Bucket '{bucket_name}' chưa tồn tại. Đang tạo mới...")
        try:
            bucket = storage_client.create_bucket(bucket_name, location=location)
            print(f"[+] Bucket '{bucket_name}' đã được tạo thành công!")
        except Conflict as e:
            print(f"[!] Lỗi: Có thể bucket đã được tạo bởi người khác hoặc tên bị trùng?")
            raise e
    except Exception as e:
        print(f"[!] Lỗi khi kiểm tra bucket: {e}")
        raise e

    bucket = storage_client.bucket(bucket_name)
    gcp_links = {}
    for filename in os.listdir(image_dir):
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        blob = bucket.blob(filename)
        local_path = os.path.join(image_dir, filename)
        blob.upload_from_filename(local_path)
        blob.make_public()  # Nếu muốn link public
        gcp_links[filename] = blob.public_url

    return gcp_links

# Example usage:
gcp_dict = upload_images_to_gcp(bucket_name="Kyanon_retrieval", image_dir="C:/Users/Admin/Data/WDM-AI-TEMIS/notebooks/Image_Retrievals/images")
print(gcp_dict)
np.savez("GCP_links.npz", **gcp_dict)


In [ ]:
# Load lại từ file
data = np.load("GCP_links.npz")
gcp_dict_loaded = {k: str(data[k]) for k in data.files}

print(gcp_dict_loaded)


In [ ]:
urls_dict = gcp_dict_loaded.values()
show_top_images_from_urls(urls_dict)

In [ ]:
from langchain.chat_models import init_chat_model
from PIL import Image
import requests
from io import BytesIO
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_google_vertexai import ChatVertexAI

# Setup Gemini
llm = init_chat_model("google_genai:gemini-2.0-flash-001")

text_encoder = SentenceTransformer("BAAI/bge-base-en")

client.recreate_collection(
                collection_name="summary_image",
                vectors_config=VectorParams(size=768, distance=Distance.COSINE)
            )


def summarize_and_index_gemini_with_document(image_url, image_id):
    message = {
    "role": "user",
    "content": [
        {
            "type": "text",
            "text": "Please analyze the image in detail. If it is a chart or graph, provide a comprehensive interpretation including the data values, axes, labels, trends, and the overall topic or subject it represents. If the image contains people or landscapes, describe all visible details such as objects, environment, actions, and potential context. Also, try to identify if any person in the image is a well-known or famous individual."
        },
        {"type": "image", "source_type": "url", "url": image_url}
    ],
}
    summary = llm.invoke([message])
    print(f"{image_id} → {summary}")

    # Tạo Document Langchain
    doc = Document(
        page_content=summary,
        metadata={
            "image_url": image_url,
            "image_id": image_id
        }
    )

   

    client.set_model(embedding_model_name="BAAI/bge-base-en")

    client.add(collection_name="summary_image", metadata=doc.metadata, documents=doc.page_content)

    return doc


# Example loop:
for fname, url in gcp_dict.items():
    summarize_and_index_gemini_with_document(url, image_id=fname)


In [ ]:


# Collection 2: embedding_image (image embedding từ CLIP)
client.recreate_collection(
                collection_name="embedding_image",
                vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
            )

In [ ]:
def embed_and_index_as_document(image_url, image_id):
    # Load image
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content)).convert("RGB")

    # Embed image
    inputs = processor(images=img, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = vlmmodel(**inputs, output_hidden_states=True)
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu().tolist()

    # Tạo Document chuẩn LangChain
    doc = Document(
        page_content="Image representation using VLM2Vec-V2.0",  # Có thể thay bằng caption sau này
        metadata={
            "image_url": image_url,
            "image_id": image_id
        }
    )

    # Index vào Qdrant
    client.upsert(
        collection_name="embedding_image",
        points=[
            {
                "id": image_id,
                "vector": embedding,
                "payload": {
                    "image_url": doc.metadata["image_url"],
                    "image_id": doc.metadata["image_id"],
                    "page_content": doc.page_content,
                },
            }
        ]
    )

    return doc

In [ ]:
for fname, url in gcp_dict.items():
    doc = embed_and_index_as_document(image_url=url, image_id=fname)

In [ ]:
from langchain.vectorstores import Qdrant

summary_retriever = Qdrant(
    client=qdrant_client,
    collection_name="summary_image",
    embeddings=text_encoder
)

vlm_retriever = Qdrant(
    client=qdrant_client,
    collection_name="embedding_image",
    embeddings=None  # không cần cho image vì đã có embedding thủ công
)

def get_documents(query: str, mode: str = "summary", top_k: int = 5) -> list[str]:
    """
    Truy vấn ảnh tương đồng với câu hỏi người dùng (văn bản), và trả về danh sách URL ảnh.

    mode:
        - "summary": so khớp với caption
        - "vlm": so khớp trực tiếp với vector ảnh (cross-modal)

    Returns:
        list[str]: danh sách image_url
    """
    if mode == "summary":
        results = summary_retriever.similarity_search_with_score(query=query, k=top_k)
        return [doc.metadata.get("image_url") for doc, _ in results]

    elif mode == "vlm":
        # Nhúng query thành vector bằng VLM
        inputs = processor(text=query, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = vlmmodel(**inputs, output_hidden_states=True)
            embedding_tensor = outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu()

        vector = embedding_tensor.tolist()

        # Truy vấn Qdrant với vector
        results = qdrant_client.search(
            collection_name="embedding_image",
            query_vector=vector,
            limit=top_k,
            with_payload=True
        )

        return [hit.payload.get("image_url") for hit in results]

    else:
        raise ValueError("mode must be 'summary' or 'vlm'")


In [ ]:
query = "Who is John Wick?"
urls = get_documents(query, mode="summary", top_k=5)
show_top_images_from_urls(urls)

In [ ]:
query = "What is sale statistics in the last year?"
urls = get_documents(query, mode="summary", top_k=5)
show_top_images_from_urls(urls)

In [ ]:
query = "Which dog is playing in the water?"
urls = get_documents(query, mode="summary", top_k=5)
show_top_images_from_urls(urls)

In [ ]:
query = "A dog playing in the water"
urls = get_documents(query, mode="vlm", top_k=5)
show_top_images_from_urls(urls)